In [3]:
import sys

print("Python:", sys.version)

Python: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


In [4]:
import pandas as pd
import requests

print("Pandas:", pd.__version__)
print("Requests:", requests.__version__)
print("Setup OK")

Pandas: 2.3.3
Requests: 2.32.5
Setup OK


In [5]:
import requests

response = requests.get("http://localhost:11434/api/tags")

print("Status:", response.status_code)

if response.status_code == 200:
    print("Ollama connection: OK")
else:
    print("Ollama connection: FAILED")

Status: 200
Ollama connection: OK


In [6]:
import requests

response = requests.get("http://localhost:11434/api/tags")

print("Installed models:")

for model in response.json()["models"]:
    print("-", model["name"])

Installed models:
- gemma3:1b


In [8]:
import os

folder = r"C:\Users\LENOVO\OneDrive - London Metropolitan University\Documents\hexagon-research"

print("CSV files found:")

for file in os.listdir(folder):
    if file.endswith(".csv"):
        print(file)

CSV files found:
Hexagon-Dataset - Education_ health.csv


In [9]:
import pandas as pd

file_path = r"C:\Users\LENOVO\OneDrive - London Metropolitan University\Documents\hexagon-research\Hexagon-Dataset - Education_ health.csv"

df = pd.read_csv(file_path)

print("File:", "Hexagon-Dataset - Education_ health.csv")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())

File: Hexagon-Dataset - Education_ health.csv
Rows: 162
Columns: 17

Columns:
['Question ID', 'Domain', 'Subdomain', 'Question Type', 'WH Type', 'Difficulty', 'English Question', 'Nepali Question', 'Gold Answer (EN)', 'Gold Answer (NE)', 'Explanation', 'Source ID', 'Source Title', 'Source URL', 'Source Type', 'Unnamed: 15', 'Created by']


In [10]:
df.head()

,Question ID,Domain,Subdomain,Question Type,WH Type,Difficulty,English Question,Nepali Question,Gold Answer (EN),Gold Answer (NE),Explanation,Source ID,Source Title,Source URL,Source Type,Unnamed: 15,Created by
0,EDU-001,Education,Higher Education,Factual,NaN,Easy,"In what year was Tribhuvan University, Nepal's...",नेपालको सबैभन्दा पुरानो विश्वविद्यालय त्रिभुवन...,1959 (25 June 1959 / 11 Ashar 2016 BS),सन् १९५९ (२०१६ साल असार ११),"Verified via web search, July 2026.",S001,Tribhuvan University official history; Britannica,https://en.wikipedia.org/wiki/Tribhuvan_Univer...,Encyclopedia/Reference,NaN,GARIMA
1,EDU-004,Education,Literacy & Access,Numerical,NaN,Medium,"According to Nepal's National Census 2021, wha...","नेपालको राष्ट्रिय जनगणना २०२१ अनुसार, ५ वर्ष म...","Approx. 76.2-76.3% (male ~83.6%, female ~69.4%)","लगभग ७६.२-७६.३% (पुरुष ~८३.६%, महिला ~६९.४%)",Minor rounding variance across reporting sourc...,S003,CBS National Census 2021 coverage,https://kathmandupost.com/national/2023/03/24/...,News/Government Data,NaN,NaN
2,EDU-005,Education,Policy/Legal,Conceptual,NaN,Medium,Under Article 31 of Nepal's Constitution (2015...,"नेपालको संविधान (२०१५) को धारा ३१ अनुसार, कुन ...",Compulsory and free basic education (up to gra...,अनिवार्य र निःशुल्क आधारभूत शिक्षा (कक्षा ८ सम...,"Verified against official constitution text, J...",S004,"Constitution of Nepal 2015, Article 31",https://nepallaws.com/Laws/constitution-of-nep...,Official Legal Text,NaN,NaN
3,EDU-007,Education,Policy/Legal,Numerical,NaN,Hard,"Per the 2018 Act, from which year will individ...","२०१८ ऐन अनुसार, कुन वर्षदेखि आधारभूत शिक्षा नभ...",From April 2028 (Nepali year 2085 BS) onward,सन् २०२८ अप्रिलदेखि (नेपाली वर्ष २०८५),"Verified via web search, July 2026.",S006,"Kathmandu Post, 'Ministry unveils blueprint to...",https://kathmandupost.com/national/2024/08/22/...,News Media,NaN,NaN
4,EDU-009,Education,Constitutional History,Factual,NaN,Hard,Which constitution first introduced Nepali cit...,नेपाली नागरिकको माध्यमिक तहसम्म निःशुल्क शिक्ष...,"The Interim Constitution of Nepal, 2007 (Artic...","नेपालको अन्तरिम संविधान, २००७ (धारा १७)","Verified via web search, July 2026.",S007,Macrothink academic journal on constitutional ...,https://www.macrothink.org/journal/index.php/i...,Academic Journal,NaN,NaN


In [11]:
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

total_chars = df.to_csv(index=False).__len__()

print("Approximate characters:", total_chars)
print("Approximate tokens:", total_chars // 4)

Number of rows: 162
Number of columns: 17
Approximate characters: 150545
Approximate tokens: 37636


In [12]:
# Convert the entire CSV into text for Ollama

dataset_context = df.to_csv(index=False)

print("Dataset characters:", len(dataset_context))
print("Approximate tokens:", len(dataset_context) // 4)

Dataset characters: 150545
Approximate tokens: 37636


In [13]:
import requests
import pandas as pd
import time

# ==========================================
# SETTINGS
# ==========================================

MODEL = "gemma3:1b"
OLLAMA_URL = "http://localhost:11434/api/generate"

# Entire dataset
full_context = df.to_csv(index=False)

print("Dataset characters:", len(full_context))
print("Approximate tokens:", len(full_context) // 4)

results = []

# ==========================================
# RUN ALL 162 QUESTIONS × 2 LANGUAGES
# ==========================================

for index, row in df.iterrows():

    question_id = row["Question ID"]

    # --------------------------------------
    # ENGLISH
    # --------------------------------------

    prompt_en = f"""
You are a dataset-only question answering system.

STRICT RULES:

1. Use ONLY information contained in the DATASET below.
2. Do NOT use your pretrained knowledge or outside information.
3. Do NOT guess.
4. If the answer cannot be found in the dataset, respond exactly: NOT FOUND.
5. Give only the answer.

DATASET:
{full_context}

QUESTION:
{row["English Question"]}

ANSWER:
"""

    try:

        response_en = requests.post(
            OLLAMA_URL,
            json={
                "model": MODEL,
                "prompt": prompt_en,
                "stream": False,
                "options": {
                    "temperature": 0
                }
            },
            timeout=600
        )

        if response_en.status_code == 200:
            answer_en = response_en.json()["response"].strip()
        else:
            answer_en = "REQUEST ERROR: " + response_en.text

    except Exception as e:
        answer_en = "REQUEST ERROR: " + str(e)

    results.append({
        "Question ID": question_id,
        "Language": "English",
        "Question": row["English Question"],
        "LLM Answer": answer_en,
        "Dataset Answer": row["Gold Answer (EN)"],
        "Manual Verdict": "",
        "Error Type": ""
    })

    print(f"{question_id} | English | Done")


    # --------------------------------------
    # NEPALI
    # --------------------------------------

    prompt_ne = f"""
तपाईं dataset मात्र प्रयोग गरेर प्रश्नको उत्तर दिने प्रणाली हुनुहुन्छ।

कडा नियमहरू:

1. तल दिइएको DATASET मा भएको जानकारी मात्र प्रयोग गर्नुहोस्।
2. आफ्नो pretrained knowledge वा बाहिरी जानकारी प्रयोग नगर्नुहोस्।
3. अनुमान नगर्नुहोस्।
4. DATASET मा उत्तर भेटिएन भने ठ्याक्कै NOT FOUND भन्नुहोस्।
5. सिधा उत्तर मात्र दिनुहोस्।

DATASET:
{full_context}

QUESTION:
{row["Nepali Question"]}

ANSWER:
"""

    try:

        response_ne = requests.post(
            OLLAMA_URL,
            json={
                "model": MODEL,
                "prompt": prompt_ne,
                "stream": False,
                "options": {
                    "temperature": 0
                }
            },
            timeout=600
        )

        if response_ne.status_code == 200:
            answer_ne = response_ne.json()["response"].strip()
        else:
            answer_ne = "REQUEST ERROR: " + response_ne.text

    except Exception as e:
        answer_ne = "REQUEST ERROR: " + str(e)

    results.append({
        "Question ID": question_id,
        "Language": "Nepali",
        "Question": row["Nepali Question"],
        "LLM Answer": answer_ne,
        "Dataset Answer": row["Gold Answer (NE)"],
        "Manual Verdict": "",
        "Error Type": ""
    })

    print(f"{question_id} | Nepali | Done")

    # Small pause
    time.sleep(0.2)


# ==========================================
# SAVE RESULTS
# ==========================================

results_df = pd.DataFrame(results)

output_file = os.path.join(
    folder,
    "Education_Health_Gemma3_1B_Results.csv"
)

results_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\n========================================")
print("EXPERIMENT COMPLETE")
print("Total responses:", len(results_df))
print("Expected responses:", len(df) * 2)
print("Saved:", output_file)
print("========================================")

Dataset characters: 150545
Approximate tokens: 37636
EDU-001 | English | Done
EDU-001 | Nepali | Done
EDU-004 | English | Done
EDU-004 | Nepali | Done
EDU-005 | English | Done
EDU-005 | Nepali | Done
EDU-007 | English | Done
EDU-007 | Nepali | Done
EDU-009 | English | Done
EDU-009 | Nepali | Done
EDU-010 | English | Done
EDU-010 | Nepali | Done
EDU-011 | English | Done
EDU-011 | Nepali | Done
EDU-012 | English | Done
EDU-012 | Nepali | Done
EDU-015 | English | Done
EDU-015 | Nepali | Done
EDU-018 | English | Done
EDU-018 | Nepali | Done
EDU-024 | English | Done
EDU-024 | Nepali | Done
EDU-025 | English | Done
EDU-025 | Nepali | Done
EDU-028 | English | Done
EDU-028 | Nepali | Done
EDU-029 | English | Done
EDU-029 | Nepali | Done
EDU-030 | English | Done
EDU-030 | Nepali | Done
EDU-031 | English | Done
EDU-031 | Nepali | Done
EDU-032 | English | Done
EDU-032 | Nepali | Done
EDU-033 | English | Done
EDU-033 | Nepali | Done
EDU-034 | English | Done
EDU-034 | Nepali | Done
EDU-035 | Englis